# ClimaTwin India — Fine-tune your custom AI model (Colab)

Fine-tunes **Qwen2.5-3B-Instruct** on training data exported from *your own* ClimaTwin twin, so the
**brain** (operator) and **guide** (friendly explainer) talk in your model's voice. QLoRA → fits a free **T4 GPU**.

**Before you start:** on your Mac, run `python scripts/export_finetune_data.py` (server running) to create
`data/finetune_all.jsonl`, and have that file handy to upload in Step 3.

Then: **Runtime ▸ Change runtime type ▸ T4 GPU**, and run every cell top to bottom.

## 1 · Check the GPU

In [ ]:
!nvidia-smi -L

## 2 · Install Unsloth (+ training deps)
Takes ~2–3 min.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps peft accelerate bitsandbytes xformers

## 3 · Upload your training data
Upload **`data/finetune_all.jsonl`** from your Mac when prompted.

In [ ]:
from google.colab import files
up = files.upload()                       # pick data/finetune_all.jsonl
DATA = list(up.keys())[0]
print("using:", DATA)

## 4 · Load Qwen2.5-3B-Instruct (4-bit) + add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth', random_state = 7,
)

## 5 · Format the chat data
Applies Qwen's chat template to each `{"messages": [...]}` line.

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files = DATA, split = 'train')
def fmt(x):
    return {'text': tokenizer.apply_chat_template(x['messages'], tokenize=False, add_generation_prompt=False)}
ds = ds.map(fmt)
print('examples:', len(ds))
print(ds[0]['text'][:400])

## 6 · Train (QLoRA, ~15–25 min on a T4)
Bump `max_steps` if you exported more data.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = 'text', max_seq_length = max_seq_length,
    dataset_num_proc = 2, packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 300, learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 10, optim = 'adamw_8bit', weight_decay = 0.01,
        lr_scheduler_type = 'linear', seed = 7, output_dir = 'outputs',
        save_strategy = 'no', report_to = 'none',   # avoid the SFTConfig pickling crash
    ),
)
trainer.train()

## 7 · Quick test
Ask it something — it should answer in the ClimaTwin voice.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {'role':'system','content':'You operate ClimaTwin India\'s digital twin (rainfall + temperature over Delhi-NCR).'},
    {'role':'user','content':'is it a good time to sow if it gets 2C warmer?'},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=160, temperature=0.3)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 8 · Export to GGUF (so Ollama can serve it)
Produces a `q4_k_m` GGUF (~2 GB).

In [ ]:
model.save_pretrained_gguf('climatwin-ft', tokenizer, quantization_method='q4_k_m')

## 9 · Download the model + a ready-to-use Modelfile
Then follow `ollama/README.md` §3 Step 2 on your Mac.

In [ ]:
import glob
ggufs = glob.glob('climatwin-ft/*.gguf') + glob.glob('*.gguf') + glob.glob('climatwin-ft*/*.gguf')
print('GGUF files:', ggufs)
gguf = ggufs[0]
name = gguf.split('/')[-1]
with open('Modelfile','w') as f:
    f.write(f'FROM ./{name}\nPARAMETER temperature 0.3\n')
from google.colab import files
files.download(gguf)
files.download('Modelfile')

## ✅ Done — finish on your Mac
```bash
mkdir -p ~/climatwin-ft && mv ~/Downloads/*.gguf ~/Downloads/Modelfile ~/climatwin-ft/
cd ~/climatwin-ft && ollama create climatwin-ft -f Modelfile
export OLLAMA_MODEL=climatwin-ft OLLAMA_GUIDE_MODEL=climatwin-ft
cd /path/to/'hack isro' && make serve     # brain + guide now use YOUR model
```
The grounding guard still verifies every number — your model improves the wording, never the facts.